In [ ]:
from flask import Flask, request, jsonify
import mysql.connector
from collections import Counter

app_notify = Flask("NotificationService")

def get_db():
    return mysql.connector.connect(host="localhost", user="root", password="", database="gadgethub_db")

# UPDATED ROUTE NAME to match Reference PDF Section 2.5
@app_notify.route('/notifications/order-confirmation', methods=['POST'])
def send_email():
    try:
        data = request.json
        order_id = data['order_id']
        email = data['email']
        customer_name = data.get('customer_name', 'Valued Customer') 
        total = data['total']
        
        # Use the pre-formatted details if passed from Order Service, else calculate
        if 'details' in data and data['details']:
            items_display_string = "\n".join(data['details'])
        else:
            # Fallback for old compatibility
            items_display_string = ", ".join(data['items'])

        # Log to DB
        conn = get_db()
        cursor = conn.cursor()
        cursor.execute("INSERT INTO notifications (order_id, email, status) VALUES (%s, %s, %s)", (order_id, email, "Sent"))
        conn.commit()
        conn.close()

        # Print Receipt
        print(f"\n==========================================")
        print(f"📧 EMAIL SIMULATION - To: {email}")
        print(f"------------------------------------------")
        print(f"🧾 RECEIPT SENT TO: {customer_name}")
        print(f"Order ID: {order_id}")
        print(f"Items Ordered:\n{items_display_string}")
        print(f"Total: ${total}")
        print("--------------------------------")
        print(f"Estimated Delivery: 3-5 Business Days.")
        print(f"Thank you for shopping at The Gadget Hub!")
        print(f"==========================================\n")

        return jsonify({"status": "Email Logged"})

    except Exception as e:
        return jsonify({"error": str(e)}), 500

if __name__ == "__main__":
    app_notify.run(port=5005, use_reloader=False)

 * Serving Flask app 'NotificationService'
 * Debug mode: off


 * Running on http://127.0.0.1:5005
Press CTRL+C to quit
127.0.0.1 - - [16/Feb/2026 14:13:05] "POST /notifications/order-confirmation HTTP/1.1" 200 -



📧 EMAIL SIMULATION - To: postman@test.com
------------------------------------------
🧾 RECEIPT SENT TO: Postman Tester
Order ID: 188875a2-f56b-4e00-80fd-1892947b57f9
Items Ordered:
Airpods :Qty 01 (from TechWorld)
Mobile Phone :Qty 01 (from GadgetCentral)
Total: $140000.0
--------------------------------
Estimated Delivery: 3-5 Business Days.
Thank you for shopping at The Gadget Hub!

